# 生命保険 アウトバウンドコール 営業成績分析

## 概要
本ノートブックは、生命保険会社のアウトバウンドコール営業データを分析し、  
**成約率向上に向けたインサイトを導出する**ことを目的としています。

### 分析対象KPI
1. 全体成績サマリー
2. 担当者別パフォーマンス分析
3. 時間帯別接触・成約率分析
4. 顧客属性（年齢・性別）× 成約率クロス分析
5. 商品別成約実績
6. 月次トレンド分析
7. チーム別比較

---
**データ**: ダミーデータ（2024年1月〜12月、5,000コール）  
**ツール**: Python / pandas / matplotlib

In [ ]:
# ライブラリインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定（文字化け防止）
import matplotlib
matplotlib.rcParams['font.family'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# カラーパレット
COLORS = {
    'primary':   '#2563EB',
    'success':   '#16A34A',
    'warning':   '#D97706',
    'danger':    '#DC2626',
    'neutral':   '#6B7280',
    'light':     '#F3F4F6',
}

print('Setup complete ✅')

In [ ]:
# データ読み込み
import os
base = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()

calls    = pd.read_csv(os.path.join(base, 'data/calls.csv'), parse_dates=['call_datetime', 'call_date'])
agents   = pd.read_csv(os.path.join(base, 'data/agents.csv'))
products = pd.read_csv(os.path.join(base, 'data/products.csv'))

# マスタ結合
df = calls.merge(agents, on='agent_id', how='left')
df = df.merge(products, on='product_id', how='left')

# フラグ列追加
df['is_contracted']  = (df['call_result'] == '成約').astype(int)
df['is_contacted']   = (df['call_result'] != '不在').astype(int)

print(f'データ読み込み完了: {len(df):,} 件')
df.head(3)

---
## 1. 全体成績サマリー

In [ ]:
total_calls      = len(df)
total_contracts  = df['is_contracted'].sum()
contract_rate    = df['is_contracted'].mean()
contact_rate     = df['is_contacted'].mean()
avg_duration     = df['call_duration_min'].mean()
contracted_dur   = df[df['is_contracted']==1]['call_duration_min'].mean()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
fig.patch.set_facecolor('#F9FAFB')

kpis = [
    ('Total Calls', f'{total_calls:,}', COLORS['primary']),
    ('Contracts', f'{total_contracts:,}', COLORS['success']),
    ('Contract Rate', f'{contract_rate:.1%}', COLORS['warning']),
    ('Contact Rate', f'{contact_rate:.1%}', COLORS['neutral']),
]

for ax, (label, value, color) in zip(axes, kpis):
    ax.set_facecolor('white')
    ax.text(0.5, 0.6, value, ha='center', va='center',
            fontsize=28, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center',
            fontsize=11, color='#374151', transform=ax.transAxes)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.05), 0.9, 0.9,
        boxstyle='round,pad=0.02', linewidth=2,
        edgecolor=color, facecolor='white', transform=ax.transAxes))

plt.suptitle('Overall KPI Summary (2024)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('kpi_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Avg call duration: {avg_duration:.1f} min  |  Contracted calls: {contracted_dur:.1f} min')

---
## 2. 担当者別パフォーマンス分析

担当者ごとのコール数・成約数・成約率を比較し、ハイパフォーマーの特徴を把握します。

In [ ]:
agent_stats = df.groupby(['agent_id', 'name', 'team', 'experience_years']).agg(
    calls=('call_id', 'count'),
    contracts=('is_contracted', 'sum'),
    avg_duration=('call_duration_min', 'mean'),
).reset_index()
agent_stats['contract_rate'] = agent_stats['contracts'] / agent_stats['calls']
agent_stats = agent_stats.sort_values('contract_rate', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#F9FAFB')

# 成約率ランキング
ax = axes[0]
ax.set_facecolor('white')
team_colors = {'東京チーム': '#2563EB', '大阪チーム': '#16A34A', '名古屋チーム': '#D97706'}
bar_colors = [team_colors[t] for t in agent_stats['team']]
bars = ax.barh(agent_stats['name'], agent_stats['contract_rate'], color=bar_colors, edgecolor='white')
ax.axvline(agent_stats['contract_rate'].mean(), color='red', linestyle='--', linewidth=1.5, label='Average')
for bar, val in zip(bars, agent_stats['contract_rate']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=9)
ax.set_xlabel('Contract Rate')
ax.set_title('Agent Contract Rate Ranking', fontweight='bold')
ax.legend()
legend_patches = [mpatches.Patch(color=c, label=t) for t, c in team_colors.items()]
ax.legend(handles=legend_patches + [plt.Line2D([0],[0], color='red', linestyle='--', label='Average')])

# 経験年数 vs 成約率 散布図
ax2 = axes[1]
ax2.set_facecolor('white')
for team, grp in agent_stats.groupby('team'):
    ax2.scatter(grp['experience_years'], grp['contract_rate'],
                color=team_colors[team], s=grp['calls']*0.3,
                alpha=0.8, label=team)
    for _, row in grp.iterrows():
        ax2.annotate(row['name'].split()[0], (row['experience_years'], row['contract_rate']),
                     textcoords='offset points', xytext=(5,3), fontsize=8)

# 回帰線
x = agent_stats['experience_years'].values
y = agent_stats['contract_rate'].values
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.max(), 100)
ax2.plot(x_line, p(x_line), 'r--', linewidth=1.5, alpha=0.7, label='Trend')
ax2.set_xlabel('Experience (years)')
ax2.set_ylabel('Contract Rate')
ax2.set_title('Experience vs Contract Rate\n(bubble size = call volume)', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig('agent_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 3 Agents:')
print(agent_stats[['name', 'team', 'calls', 'contracts', 'contract_rate']].head(3).to_string(index=False))

---
## 3. 時間帯別 接触率・成約率分析

どの時間帯にコールすると効果的かを分析し、最適なコールタイミングを特定します。

In [ ]:
hourly = df.groupby('call_hour').agg(
    calls=('call_id', 'count'),
    contracts=('is_contracted', 'sum'),
    contacted=('is_contacted', 'sum'),
).reset_index()
hourly['contract_rate'] = hourly['contracts'] / hourly['calls']
hourly['contact_rate']  = hourly['contacted'] / hourly['calls']

fig, ax1 = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#F9FAFB')
ax1.set_facecolor('white')

bar_colors = [COLORS['success'] if r == hourly['contract_rate'].max() else COLORS['primary']
              for r in hourly['contract_rate']]
bars = ax1.bar(hourly['call_hour'], hourly['contract_rate'], color=bar_colors, alpha=0.8, width=0.6)
ax1.set_xlabel('Call Hour')
ax1.set_ylabel('Contract Rate', color=COLORS['primary'])
ax1.set_xticks(hourly['call_hour'])
ax1.set_xticklabels([f'{h}:00' for h in hourly['call_hour']])

ax2 = ax1.twinx()
ax2.plot(hourly['call_hour'], hourly['contact_rate'], 'o-',
         color=COLORS['warning'], linewidth=2.5, markersize=7, label='Contact Rate')
ax2.set_ylabel('Contact Rate', color=COLORS['warning'])

best_hour = hourly.loc[hourly['contract_rate'].idxmax(), 'call_hour']
ax1.axvline(best_hour, color='red', linestyle='--', alpha=0.5)
ax1.text(best_hour + 0.1, ax1.get_ylim()[1]*0.95, f'Best: {best_hour}:00',
         color='red', fontsize=10, fontweight='bold')

plt.title('Contract Rate & Contact Rate by Hour of Day', fontweight='bold', fontsize=13)
ax1.legend(['Contract Rate (bar)'], loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig('hourly_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. 顧客属性クロス分析（年齢 × 性別 × 成約率）

In [ ]:
cross = df.groupby(['customer_age_group', 'customer_gender']).agg(
    calls=('call_id', 'count'),
    contract_rate=('is_contracted', 'mean'),
).reset_index()

age_order = ['20代', '30代', '40代', '50代', '60代以上']
pivot = cross.pivot(index='customer_age_group', columns='customer_gender', values='contract_rate')
pivot = pivot.reindex(age_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#F9FAFB')

# ヒートマップ（手動実装）
ax = axes[0]
ax.set_facecolor('white')
data = pivot.values
im = ax.imshow(data, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j, i, f'{data[i,j]:.1%}', ha='center', va='center',
                fontweight='bold', fontsize=13,
                color='white' if data[i,j] > data.max()*0.6 else 'black')
plt.colorbar(im, ax=ax, label='Contract Rate')
ax.set_title('Contract Rate Heatmap\n(Age Group × Gender)', fontweight='bold')

# 年齢別 棒グラフ
ax2 = axes[1]
ax2.set_facecolor('white')
x = np.arange(len(age_order))
width = 0.35
for i, gender in enumerate(pivot.columns):
    color = COLORS['primary'] if gender == '男性' else COLORS['warning']
    bars = ax2.bar(x + i*width - width/2, pivot[gender].values, width,
                   label=gender, color=color, alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(age_order)
ax2.set_ylabel('Contract Rate')
ax2.set_title('Contract Rate by Age Group & Gender', fontweight='bold')
ax2.legend()
ax2.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))

plt.tight_layout()
plt.savefig('customer_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. 商品別 成約実績

In [ ]:
product_stats = df[df['is_contracted']==1].groupby('product_name').agg(
    contracts=('call_id', 'count'),
    monthly_premium=('monthly_premium', 'first'),
).reset_index()
product_stats['estimated_revenue'] = product_stats['contracts'] * product_stats['monthly_premium'] * 12
product_stats = product_stats.sort_values('contracts', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#F9FAFB')

pal = [COLORS['primary'], COLORS['success'], COLORS['warning'],
       COLORS['danger'], COLORS['neutral'], '#7C3AED']

ax = axes[0]
ax.set_facecolor('white')
bars = ax.barh(product_stats['product_name'], product_stats['contracts'],
               color=pal[:len(product_stats)])
for bar, val in zip(bars, product_stats['contracts']):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=10)
ax.set_xlabel('Number of Contracts')
ax.set_title('Contracts by Product', fontweight='bold')

ax2 = axes[1]
ax2.set_facecolor('white')
rev_sorted = product_stats.sort_values('estimated_revenue')
bars2 = ax2.barh(rev_sorted['product_name'], rev_sorted['estimated_revenue'] / 1e6,
                 color=pal[:len(rev_sorted)])
for bar, val in zip(bars2, rev_sorted['estimated_revenue']):
    ax2.text(val/1e6 + 0.01, bar.get_y() + bar.get_height()/2,
             f'¥{val/1e6:.1f}M', va='center', fontsize=9)
ax2.set_xlabel('Estimated Annual Revenue (million yen)')
ax2.set_title('Estimated Annual Revenue by Product', fontweight='bold')

plt.tight_layout()
plt.savefig('product_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. 月次トレンド分析

In [ ]:
monthly = df.groupby('call_month').agg(
    calls=('call_id', 'count'),
    contracts=('is_contracted', 'sum'),
    contract_rate=('is_contracted', 'mean'),
).reset_index().sort_values('call_month')

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.patch.set_facecolor('#F9FAFB')

months = range(len(monthly))
labels = [m[5:] + '月' for m in monthly['call_month']]

ax1 = axes[0]
ax1.set_facecolor('white')
ax1.bar(months, monthly['calls'], color=COLORS['primary'], alpha=0.6, label='Total Calls')
ax1.bar(months, monthly['contracts'], color=COLORS['success'], alpha=0.9, label='Contracts')
ax1.set_xticks(months)
ax1.set_xticklabels(labels)
ax1.set_ylabel('Count')
ax1.set_title('Monthly Call Volume & Contracts', fontweight='bold')
ax1.legend()

ax2 = axes[1]
ax2.set_facecolor('white')
ax2.plot(months, monthly['contract_rate'], 'o-', color=COLORS['warning'],
         linewidth=2.5, markersize=8)
ax2.fill_between(months, monthly['contract_rate'], alpha=0.15, color=COLORS['warning'])
ax2.axhline(monthly['contract_rate'].mean(), color='red', linestyle='--',
            linewidth=1.5, label=f"Avg: {monthly['contract_rate'].mean():.1%}")
ax2.set_xticks(months)
ax2.set_xticklabels(labels)
ax2.set_ylabel('Contract Rate')
ax2.set_title('Monthly Contract Rate Trend', fontweight='bold')
ax2.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax2.legend()

plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. まとめ・インサイト

### 🎯 分析から得られた主要インサイト

| # | インサイト | アクション提案 |
|---|-----------|---------------|
| 1 | 経験年数と成約率に正の相関 | 新人エージェントへのメンタリング強化 |
| 2 | 特定の時間帯に成約率が集中 | コールスケジュールの最適化 |
| 3 | 40〜50代は成約率が高い傾向 | ターゲットリストの優先度付け |
| 4 | 個人年金・終身保険は推定収益が高い | 重点商品の研修強化 |
| 5 | 月次で成約率に波がある | 季節要因の特定と対策立案 |

### 📌 次のステップ
- より多くの期間データでのトレンド分析
- 機械学習による成約予測モデルの構築
- A/Bテスト設計（トークスクリプト最適化）

In [ ]:
print('=' * 50)
print('Analysis Complete ✅')
print('=' * 50)
print(f'Total calls analyzed : {len(df):,}')
print(f'Contract rate        : {df["is_contracted"].mean():.1%}')
print(f'Top agent            : {agent_stats.iloc[0]["name"]} ({agent_stats.iloc[0]["contract_rate"]:.1%})')
print(f'Best hour            : {hourly.loc[hourly["contract_rate"].idxmax(), "call_hour"]}:00')
print('Output images saved  : kpi_summary, agent_performance, hourly_analysis, customer_analysis, product_analysis, monthly_trend')